In [1]:
# importing Models and metrics

from sklearn.linear_model import LinearRegression

from sklearn.neighbors import KNeighborsRegressor

from sklearn.svm import SVR

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
r2_score
)

import numpy as np
import pandas as pd

In [2]:
#Creating evaluation function

def evaluate_model(model,X_train,y_train,X_valid,y_valid):

    model.fit(X_train,y_train)

    pred=model.predict(X_valid)

    mae=mean_absolute_error(
        y_valid,
        pred
    )

    rmse=np.sqrt(
        mean_squared_error(
            y_valid,
            pred
        )
    )

    r2=r2_score(
        y_valid,
        pred
    )

    return mae,rmse,r2

In [ ]:
#importing processed data

import pickle

with open("artifacts/X_train_processed.pkl","rb") as f:
    X_train_processed=pickle.load(f)

with open("artifacts/X_test_processed.pkl","rb") as f:
    X_test_processed=pickle.load(f)

with open("artifacts/y_train.pkl","rb") as f:
    y_train=pickle.load(f)

with open("artifacts/y_test.pkl","rb") as f:
    y_test=pickle.load(f)

print(X_train_processed.shape)
print(X_test_processed.shape)
print(y_train.shape)
print(y_test.shape)

(960000, 41)
(240000, 41)
(960000,)
(240000,)


In [4]:

#for KNN ang SVR we will use a sample of the data to speed up the training process

sample_X=X_train_processed[:50000]

sample_y=y_train.iloc[:50000]

In [5]:
import os


from pathlib import Path


import mlflow


import mlflow.sklearn




project_root = Path(r"E:\Laptop\mini project 3")


mlflow_db = project_root / "mlflow.db"


tracking_uri = f"sqlite:///{mlflow_db.as_posix()}"




# If the DB is corrupted, back it up before rerunning.


# mlflow_db.rename(project_root / "mlflow.db.bak")




mlflow.set_tracking_uri(tracking_uri)


mlflow.set_experiment("SmartPremium_Regression")

c:\Users\ADMIN\anaconda3\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


<Experiment: artifact_location='file:e:/Laptop/mini project 3/Notebooks/mlruns/1', creation_time=1779980455764, experiment_id='1', last_update_time=1779980455764, lifecycle_stage='active', name='SmartPremium_Regression', tags={}, workspace='default'>

In [6]:


# creating mlflow training function
def train_and_log(
        model,
        model_name,
        X_train,
        y_train,
        X_valid,
        y_valid):


    with mlflow.start_run(
            run_name=model_name):


        model.fit(
            X_train,
            y_train
        )

        pred=model.predict(
            X_valid
        )


        mae=mean_absolute_error(
            y_valid,
            pred
        )

        rmse=np.sqrt(
            mean_squared_error(
                y_valid,
                pred
            )
        )

        r2=r2_score(
            y_valid,
            pred
        )


        mlflow.log_metric(
            "MAE",
            mae
        )

        mlflow.log_metric(
            "RMSE",
            rmse
        )

        mlflow.log_metric(
            "R2",
            r2
        )


        params=model.get_params()

        for key,value in params.items():

            try:
                mlflow.log_param(
                    key,
                    value
                )

            except:
                pass


        mlflow.sklearn.log_model(
            model,
            model_name
        )


        print(model_name)

        print("MAE:",mae)

        print("RMSE:",rmse)

        print("R2:",r2)

        return mae,rmse,r2



In [7]:
#models

models={

"Linear Regression":
LinearRegression(),

"KNN":
KNeighborsRegressor(
n_neighbors=5
),

"SVR":
SVR(),

"Decision Tree":
DecisionTreeRegressor(
random_state=42
),

"Random Forest":
RandomForestRegressor(
n_estimators=100,
random_state=42,
n_jobs=-1
),

"XGBoost":
XGBRegressor(
n_estimators=100,
random_state=42,
n_jobs=-1
)
}

In [ ]:
#training and logging models

results=[]

for name,model in models.items():

    print("Training:",name)

    if name in ["KNN","SVR"]:

        mae,rmse,r2=train_and_log(

            model,

            name,

            sample_X,

            sample_y,

            X_test_processed,

            y_test
        )

    else:

        mae,rmse,r2=train_and_log(

            model,

            name,

            X_train_processed,

            y_train,

            X_test_processed,

            y_test
        )


    results.append(
        [name,mae,rmse,r2]
    )

2026/05/29 11:58:22 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Training: Linear Regression


2026/05/29 11:58:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 11:58:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression
MAE: 667.267622470506
RMSE: 863.2710140895872
R2: 0.002741578849028592
Training: KNN


2026/05/29 11:58:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 11:58:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

Decision Tree
MAE: 897.4155375
RMSE: 1232.780136475073
R2: -1.0336895199187972
Training: Random Forest


2026/05/29 12:42:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 12:42:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest
MAE: 656.0082401249997
RMSE: 852.5420156916522
R2: 0.02737600990122213
Training: XGBoost


2026/05/29 12:44:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 12:44:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost
MAE: 642.4245500220537
RMSE: 844.8365890429669
R2: 0.04487805540655809


In [9]:
#Create comparison table


results_df=pd.DataFrame(
results,
columns=[
"Model",
"MAE",
"RMSE",
"R2"
]
)

results_df.sort_values(
"RMSE"
)

,Model,MAE,RMSE,R2
5,XGBoost,642.424550,844.836589,0.044878
4,Random Forest,656.008240,852.542016,0.027376
0,Linear Regression,667.267622,863.271014,0.002742
2,SVR,639.230631,895.997696,-0.074304
1,KNN,719.817281,939.618327,-0.181453
3,Decision Tree,897.415538,1232.780136,-1.033690


In [ ]:
#Hyperperameter Tuning for Random Forest

from sklearn.model_selection import RandomizedSearchCV

rf=RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

rf_params={

'n_estimators':[100,200,300],

'max_depth':[10,20,30,None],

'min_samples_split':[2,5,10],

'min_samples_leaf':[1,2,4]
}

rf_search=RandomizedSearchCV(

estimator=rf,

param_distributions=rf_params,

n_iter=10,

cv=3,

scoring='neg_root_mean_squared_error',

verbose=2,

n_jobs=-1,

random_state=42
)

rf_search.fit(
    X_train_processed[:100000],
    y_train.iloc[:100000]
)

print(rf_search.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 10}


In [ ]:
#Hyperparameter tuning for XGBoost

xgb=XGBRegressor(
    random_state=42,
    n_jobs=-1
)

xgb_params={

'n_estimators':[100,200,300],

'max_depth':[3,5,7],

'learning_rate':[0.01,0.05,0.1],

'subsample':[0.8,1],

'colsample_bytree':[0.8,1]
}

xgb_search=RandomizedSearchCV(

xgb,

param_distributions=xgb_params,

n_iter=10,

cv=3,

scoring='neg_root_mean_squared_error',

verbose=2,

n_jobs=-1,

random_state=42
)

xgb_search.fit(
    X_train_processed[:100000],
    y_train.iloc[:100000]
)

print(xgb_search.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


In [ ]:
#Evaluate the best Random Forest model on the test set

best_rf = rf_search.best_estimator_

pred_rf = best_rf.predict(
    X_test_processed
)

rf_mae = mean_absolute_error(
    y_test,
    pred_rf
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_rf
    )
)

rf_r2 = r2_score(
    y_test,
    pred_rf
)




In [ ]:
#Evaluate the best XGBoost model on the test set

with mlflow.start_run(
    run_name="Tuned_RandomForest"
):

    mlflow.log_params(
        rf_search.best_params_
    )

    mlflow.log_metric(
        "MAE",
        rf_mae
    )

    mlflow.log_metric(
        "RMSE",
        rf_rmse
    )

    mlflow.log_metric(
        "R2",
        rf_r2
    )

    mlflow.sklearn.log_model(
        best_rf,
        "RandomForest"
    )

2026/05/29 12:46:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 12:46:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
#Log the best XGBoost model

best_xgb = xgb_search.best_estimator_

pred_xgb = best_xgb.predict(
    X_test_processed
)

xgb_mae = mean_absolute_error(
    y_test,
    pred_xgb
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_xgb
    )
)

xgb_r2 = r2_score(
    y_test,
    pred_xgb
)

In [ ]:
#

with mlflow.start_run(
    run_name="Tuned_XGBoost"
):

    mlflow.log_params(
        xgb_search.best_params_
    )

    mlflow.log_metric(
        "MAE",
        xgb_mae
    )

    mlflow.log_metric(
        "RMSE",
        xgb_rmse
    )

    mlflow.log_metric(
        "R2",
        xgb_r2
    )

    mlflow.sklearn.log_model(
        best_xgb,
        "XGBoost"
    )

2026/05/29 12:46:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 12:46:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [19]:
best_model = best_rf

In [ ]:
#Create artifacts directory if it doesn't exist
import os

os.makedirs(
    "artifacts",
    exist_ok=True
)


In [ ]:
#Save the best model as a pickle file
import pickle

with open(
    "artifacts/final_model.pkl",
    "wb"
) as f:

    pickle.dump(
        best_model,
        f
    )

print("Model saved")

Model saved


In [ ]:
#List artifacts directory contents
import os

print(
    os.listdir("artifacts")
)

['final_model.pkl', 'preprocessor.pkl']


In [ ]:
#Load the model from the pickle file and check its type

with open(
    "artifacts/final_model.pkl",
    "rb"
) as f:

    loaded_model = pickle.load(f)

print(type(loaded_model))

<class 'sklearn.ensemble._forest.RandomForestRegressor'>
